<a href="https://colab.research.google.com/github/Rayoyo/NLP-Translator-JA-EN/blob/main/nlp_translator_main_VER10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP translator - *Natural Language Processing*
This notebook represents a comprehensive pipeline for building, training, and evaluating a **Neural Machine Translation (NMT)** system based on the Transformer architecture for Japanese-English translation

## 1. Enviroment setup & Dependency Management
This section ensures the hardware is capable of the heavy lifting required for Deep Learning.

- ***GPU Verification:*** It checks for a CUDA-compatible GPU (specifically a Tesla T4 in this instance). \
This is critical because Transformers rely on parallel matrix multiplications that are roughly 50-100x faster on a GPU than a CPU.

- ***Keep-Alive Script:*** A clever JavaScript hack that simulates a click on the Colab "Connect" button every 60 seconds. \
This prevents the Google Colab backend from timing out due to "inactivity" during long training loops.

- ***Library Installation:*** Installs sentencepiece (for subword tokenization), sacrebleu (the industry standard for NMT evaluation), and gradio (for the final UI).



In [1]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# To keep Colab "alive" avoiding inactivity timeout
import time
from IPython.display import display, Javascript

def keep_alive():
    display(Javascript('''
        function keepAlive() {
            setInterval(() => {
                document.querySelector("colab-toolbar-button#connect").click();
                console.log("Keep alive");
            }, 60000);
        }
        keepAlive();
    '''))

keep_alive()

<IPython.core.display.Javascript object>

In [3]:
# Install dependencies
!pip install -q sentencepiece sacrebleu transformers gradio datasets tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import torch
import torch.nn as nn
import sys
import os

---
## 2. Clone repository & File System Integration

The notebook connects to external storage to keep progress persistent.

- ***GitHub Cloning:*** It clones the custom source code (`src/`) which contains the modular logic for the Transformer, training loops, and data handling;

- ***Google Drive Mounting:*** Because *Colab's* local disk is wiped after every session, *Google Drive* is used to store the massive dataset files (`english.txt`, `japanese.txt`) and the model weights (`.pt files`)

In [6]:
# Clone repo (or load manually files on Colab)
# !git clone https://github.com/Rayoyo/NLP-Translator-JA-EN.git     # for first run
!git clone https://github.com/Rayoyo/NLP-Translator-JA-EN.git
%cd NLP-Translator-JA-EN

import sys
sys.path.append('/content/NLP-Translator-JA-EN')

print("Repo cloned!")
!ls -la src/

Cloning into 'NLP-Translator-JA-EN'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 203 (delta 0), reused 0 (delta 0), pack-reused 202 (from 1)
Receiving objects: 100% (203/203), 2.36 MiB | 16.90 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/content/NLP-Translator-JA-EN
Repo cloned!
total 68
drwxr-xr-x 2 root root  4096 Jun  4 23:13 .
drwxr-xr-x 6 root root  4096 Jun  4 23:13 ..
-rw-r--r-- 1 root root  6641 Jun  4 23:13 dataset.py
-rw-r--r-- 1 root root  5094 Jun  4 23:13 evaluate.py
-rw-r--r-- 1 root root  2957 Jun  4 23:13 gui.py
-rw-r--r-- 1 root root     0 Jun  4 23:13 __init__.py
-rw-r--r-- 1 root root  2370 Jun  4 23:13 tokenizer.py
-rw-r--r-- 1 root root 12948 Jun  4 23:13 train.py
-rw-r--r-- 1 root root 19805 Jun  4 23:13 transformer.py


---
## 3. Hyperparameter Configuration
It defines the model's capacity and the training strategy:
| Parameter | Number | Purpose |
| :--- | :--- | :--- |
| `VOCAB_SIZE` | 32K | Defines how many unique subwords the model can recognize in each language |
| `BATCH_SIZE` | 4 | Number of training examples processed at once. It is kept low to avoid "Out of Memory" (OOM) errors on the GPU |
| `ACCUMULATOR_STEPS` | 4 | Compensates for the small batch size by summing gradients over 4 steps before updating weights, effectively creating a batch size of 16 |
| `D_MODEL` | 512 | The size of the vector embedding for each word - higher value captures more nuance, requiring more memory |
| `N_HEADS` | 8 | The number of "Self-Attention" heads, allowing the model to focus on different parts of a sentence simultaneously |
| `N_LAYERS` | 6 | The number of encoder and decoder blocks stacked on top of each other |
| `N_HEADS` | 8 | The number of attention heads allowing the model to focus on different parts of a sentence (e.g., syntax vs. semantics) |
| `D_FF` | 2048| The dimensionality of the inner "Feed-Forward" layers |
| `MAX_SAMPLES` | 100/200K | Limits the dataset size to speed up experimentation |


In [7]:
PROJECT_PATH = "/content/drive/MyDrive/University/Project-NLP_Translator"
DATA_PATH = f"{PROJECT_PATH}/data/processed"
MODELS_PATH = f"{PROJECT_PATH}/models"

os.makedirs(MODELS_PATH, exist_ok=True)

EN_FILE = f"{DATA_PATH}/english.txt"
JP_FILE = f"{DATA_PATH}/japanese.txt"

#print(f"EN: {os.path.exists(EN_FILE)} ({os.path.getsize(EN_FILE)/1e9:.2f} GB)")
#print(f"JP: {os.path.exists(JP_FILE)} ({os.path.getsize(JP_FILE)/1e9:.2f} GB)")

# Parameters
VOCAB_SIZE = 32000
BATCH_SIZE = 4          # 32 OUT OF MEMORY
ACCUMULATOR_STEPS = 4
D_MODEL = 512
N_HEADS = 8
N_LAYERS = 6
D_FF = 2048
MAX_SAMPLES = 100_000   # None = all dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Batch GPU: {BATCH_SIZE}")
print(f"Accumulation: {ACCUMULATOR_STEPS}")
print(f"Effective batch: {BATCH_SIZE * ACCUMULATOR_STEPS}")
print(f"Max samples: {MAX_SAMPLES:,}")
print(f"Batch per epoch: {MAX_SAMPLES // BATCH_SIZE:,}")

Batch GPU: 4
Accumulation: 4
Effective batch: 16
Max samples: 100,000
Batch per epoch: 25,000


---
## 4. Path Validation & Tokenizer Loading

Before training, the notebook verifies the integrity of the data.

- ***Path Verify:*** A safety check to ensure that `english.txt` and `japanese.txt` are correctly located in *Drive* before starting the resource-intensive training

- ***SentencePiece Tokenizers:*** Loads pre-trained models for tokenization. \
This is vital for Japanese translation because it breaks down text into meaningful sub-units rather than just spaces, which Japanese lacks


In [8]:
#File search on drive
'''
import os

print("🔍 Searching english.txt e japanese.txt in all Drive...")
print("=" * 60)

found_en = []
found_jp = []

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        full_path = os.path.join(root, file)
        if file == "english.txt":
            found_en.append(full_path)
            size = os.path.getsize(full_path) / 1e9
            print(f"✅ FOUND english.txt:")
            print(f"   Path: {full_path}")
            print(f"   Size: {size:.2f} GB")
            print()
        elif file == "japanese.txt":
            found_jp.append(full_path)
            size = os.path.getsize(full_path) / 1e9
            print(f"✅ FOUND japanese.txt:")
            print(f"   Path: {full_path}")
            print(f"   Size: {size:.2f} GB")
            print()

print("=" * 60)
if not found_en:
    print("❌ english.txt NOT FOUND in Drive")
if not found_jp:
    print("❌ japanese.txt NOT FOUND in Drive")

# Searching for similar names
print("\n🔍 File .txt big (>1GB) on Drive:")
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.txt'):
            full = os.path.join(root, file)
            try:
                size = os.path.getsize(full)
                if size > 1e9:
                    print(f"   {size/1e9:.2f} GB  ->  {full}")
            except:
                pass
'''

'\nimport os\n\nprint("🔍 Searching english.txt e japanese.txt in all Drive...")\nprint("=" * 60)\n\nfound_en = []\nfound_jp = []\n\nfor root, dirs, files in os.walk(\'/content/drive/MyDrive\'):\n    for file in files:\n        full_path = os.path.join(root, file)\n        if file == "english.txt":\n            found_en.append(full_path)\n            size = os.path.getsize(full_path) / 1e9\n            print(f"✅ FOUND english.txt:")\n            print(f"   Path: {full_path}")\n            print(f"   Size: {size:.2f} GB")\n            print()\n        elif file == "japanese.txt":\n            found_jp.append(full_path)\n            size = os.path.getsize(full_path) / 1e9\n            print(f"✅ FOUND japanese.txt:")\n            print(f"   Path: {full_path}")\n            print(f"   Size: {size:.2f} GB")\n            print()\n\nprint("=" * 60)\nif not found_en:\n    print("❌ english.txt NOT FOUND in Drive")\nif not found_jp:\n    print("❌ japanese.txt NOT FOUND in Drive")\n\n# Searching f

In [9]:
print("=" * 50)
print("PATH VERIFY")
print("=" * 50)

# Check if file exists
en_exists = os.path.exists(EN_FILE)
jp_exists = os.path.exists(JP_FILE)

print(f"EN file: {EN_FILE}")
print(f"  Exists: {en_exists}")
if en_exists:
    print(f"  Dimension: {os.path.getsize(EN_FILE)/1e9:.2f} GB")

print(f"JP file: {JP_FILE}")
print(f"  Exists: {jp_exists}")
if jp_exists:
    print(f"  Dimension: {os.path.getsize(JP_FILE)/1e9:.2f} GB")

print(f"\nModels path: {MODELS_PATH}")
print(f"Device: {DEVICE}")

# Blocks everything if files do not exist
if not en_exists or not jp_exists:
    raise FileNotFoundError(
        "❌ Files not found on Drive\n"
        "Verify that english.txt and japanese.txt are in:\n"
        f"{DATA_PATH}"
    )

print("\n✅ Every path is correct!")

PATH VERIFY
EN file: /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/english.txt
  Exists: True
  Dimension: 0.18 GB
JP file: /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/japanese.txt
  Exists: True
  Dimension: 0.23 GB

Models path: /content/drive/MyDrive/University/Project-NLP_Translator/models
Device: cuda

✅ Every path is correct!


---
## 5. Lazy Dataset Loading

To handle massive datasets without crashing the system's RAM, the notebook uses a Lazy Loader.

- ***Offset Indexing:*** Instead of loading 1GB of text into RAM (which would crash Colab), the `LazyTranslationDataset` uses byte offsets (the exact location in the file) for every line.

- ***Binary Seek:*** It maps the starting position of every line in the file. \
When the model asks for a specific sentence, the code "jumps" directly to that byte in the file, reads only that line, and tokenizes it on the fly

In [10]:
from src.dataset import create_dataloaders
from src.transformer import Transformer, count_parameters
from src.train import Trainer, get_scheduler

print("✅ Moduls import success!")

✅ Moduls import success!


In [11]:
import sentencepiece as spm

# Load existing tokenizer in local
sp_en = spm.SentencePieceProcessor(model_file=f"{MODELS_PATH}/en_tokenizer.model")
sp_jp = spm.SentencePieceProcessor(model_file=f"{MODELS_PATH}/jp_tokenizer.model")

print(f"English tokenizer vocab: {sp_en.get_piece_size():,}")
print(f"Japanese tokenizer vocab: {sp_jp.get_piece_size():,}")

# Test rapido
test_en = "Hello, how are you?"
test_jp = "今日は良い天気ですね。"

en_ids = sp_en.encode(test_en, out_type=int, add_bos=True, add_eos=True)
jp_ids = sp_jp.encode(test_jp, out_type=int, add_bos=True, add_eos=True)

print(f"\nTest EN: '{test_en}'")
print(f"  Token IDs: {en_ids}")
print(f"  Decoded:   {sp_en.decode(en_ids)}")

print(f"\nTest JP: '{test_jp}'")
print(f"  Token IDs: {jp_ids}")
print(f"  Decoded:   {sp_jp.decode(jp_ids)}")

English tokenizer vocab: 32,000
Japanese tokenizer vocab: 32,000

Test EN: 'Hello, how are you?'
  Token IDs: [1, 8642, 4, 122, 21, 18, 71, 2]
  Decoded:   Hello, how are you?

Test JP: '今日は良い天気ですね。'
  Token IDs: [1, 12710, 548, 7945, 2912, 5, 2]
  Decoded:   今日は良い天気ですね。


---
## 6. Model Architecture & Training Patching

This part prepares the Transformer for the training phase.

*The Patching Logic* - The notebook includes scripts to modify the raw `.py` files on the fly.

- ***Numerical Stability:*** It changes a masking value from `-1e9` to `-1e4`. \
In "Mixed Precision" training (FP16), `-1e9` is too large and can cause "NaN" (Not a Number) errors.

- ***AMP (Automatic Mixed Precision):*** It updates the code to use `torch.amp`, which allows the model to use `16-bit floats` where possible, nearly doubling training speed on the `Tesla T4`

In [12]:
# Estrai test set PRIMA di creare il dataloader (escludendo quegli indici)
# Oppure semplicemente usa file separati per train/test

In [13]:
from src.dataset import create_dataloaders

train_loader = create_dataloaders(
    EN_FILE,
    JP_FILE,
    sp_en,
    sp_jp,
    batch_size=BATCH_SIZE,
    num_workers=0,
    max_samples=MAX_SAMPLES  # For sanity check: at the start use 100000
)

print(f"Train batches: {len(train_loader)}")

# Test: take a batch
batch_src, batch_tgt = next(iter(train_loader))
print(f"\nSource bach shape (EN): {batch_src.shape}")
print(f"Targhet bach shape (JP):   {batch_tgt.shape}")
print(f"Example src IDs: {batch_src[0][:15]}...")
print(f"Example tgt IDs: {batch_tgt[0][:15]}...")

LazyDataset: indexed 100,000 lines (RAM usage: ~0.8 MB)
Train batches: 25000

Source bach shape (EN): torch.Size([4, 70])
Targhet bach shape (JP):   torch.Size([4, 128])
Example src IDs: tensor([    1,  4773,     4,   121,  3113,   130,   224,   106,     4, 22106,
            6,   133,  5043,  3090,     5])...
Example tgt IDs: tensor([    1,  9177, 11418,   735,  3572,  1188,    21,     2,     0,     0,
            0,     0,     0,     0,     0])...


In [14]:
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_encoder_layers=N_LAYERS,
    n_decoder_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=0.1,
    pad_idx=0
).to(DEVICE)

print(f"Model created!")
print(f"Total parameters: {count_parameters(model):,}")
print(f"Device: {DEVICE}")

Model created!
Total parameters: 93,322,496
Device: cuda


In [15]:
# Optimizer & scheduler
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)
scheduler = get_scheduler(optimizer, D_MODEL, warmup_steps=2000)

---
## 7. Training & Auto-Resume

- ***Warmup Scheduler:*** The learning rate starts very low and increases for the first `2,000` steps. \
This *"warms up"* the weights to prevent the model from failing due to early mathematical instability.

- ***Check-pointing:*** Includes a function to find the latest saved epoch in Drive and automatically resume from where it left off.

- ***The Fit Method:*** This is the core training loop where the model performs the forward pass, calculates loss (error), and performs backpropagation to learn.

In [16]:
# Fix per transformer.py: -1e9 overflowa in float16 (mixed precision)
# Sostituisce con -1e4 che è sufficiente per softmax

import re

transformer_path = '/content/NLP-Translator-JA-EN/src/transformer.py'

with open(transformer_path, 'r') as f:
    content = f.read()

# Sostituisci -1e9 con -1e4 nel masking
content = content.replace("scores.masked_fill(mask == 0, -1e9)",
                          "scores.masked_fill(mask == 0, -1e4)")

with open(transformer_path, 'w') as f:
    f.write(content)

print("✅ transformer.py patchato")
!grep -n "masked_fill" /content/NLP-Translator-JA-EN/src/transformer.py

✅ transformer.py patchato
98:            scores = scores.masked_fill(mask == 0, -1e4)
101:            masked_fill → sets those positions to a very large negative value (-1e4) 


In [17]:
# Fix for train.py: updates autocast and GradScaler

train_path = '/content/NLP-Translator-JA-EN/src/train.py'

with open(train_path, 'r') as f:
    content = f.read()

# Aggiorna import
content = content.replace(
    "from torch.cuda.amp import autocast, GradScaler",
    "from torch.amp import autocast, GradScaler"
)

# Aggiorna GradScaler
content = content.replace(
    "self.scaler = GradScaler()",
    "self.scaler = GradScaler('cuda')"
)

# Aggiorna autocast
content = content.replace(
    "with autocast():",
    "with autocast('cuda'):"
)

with open(train_path, 'w') as f:
    f.write(content)

print("✅ train.py patched")
!grep -n "GradScaler\|autocast" /content/NLP-Translator-JA-EN/src/train.py

✅ train.py patched
9:from torch.amp import autocast, GradScaler
49:        self.scaler = GradScaler('cuda')         # Mixed precision
79:            with autocast('cuda'):
157:            with autocast('cuda'):


In [18]:
import importlib
import sys

# Remove cache
for mod in list(sys.modules.keys()):
    if 'src.' in mod:
        del sys.modules[mod]

# Re-import
from src.transformer import Transformer, count_parameters
from src.train import Trainer, get_scheduler

print("Moduls reloaded wioth fix")

Moduls reloaded wioth fix


In [19]:
'''
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_encoder_layers=N_LAYERS,
    n_decoder_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=0.1,
    pad_idx=0
).to(DEVICE)
'''

# Verify before training
batch_src, batch_tgt = next(iter(train_loader))
with torch.no_grad():
    tgt_input = batch_tgt[:, :-1].to(DEVICE)
    out = model(batch_src.to(DEVICE), tgt_input)
    loss = nn.CrossEntropyLoss(ignore_index=0)(
        out.reshape(-1, VOCAB_SIZE),
        batch_tgt[:, 1:].reshape(-1).to(DEVICE)
    )
print(f"\nLoss first batch (no training): {loss.item():.2f}")
print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
print(f"Predicted loss ~10-12, LR ~5e-4")



print(f"Parameters: {count_parameters(model):,}")

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4, betas=(0.9, 0.98), eps=1e-9)
# scheduler = get_scheduler(optimizer, D_MODEL, warmup_steps=2000) -> lr constant and more secure

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    scheduler=None,
    device=DEVICE,
    save_dir=MODELS_PATH,
    log_interval=50,
    accumulator_steps=ACCUMULATOR_STEPS
)

N_EPOCHS = 10

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/University/Project-NLP_Translator/data/processed/english.txt'

In [ ]:
# To cancel previous epochs backups in case of parameter tuning
'''
import glob, os
old_cps = glob.glob(f"{MODELS_PATH}/checkpoint_epoch_*.pt")
for cp in old_cps:
    os.remove(cp)
    print(f"Rimosso: {cp}")
'''

In [20]:
# ============================================================
# AUTO-RESUME: finds last checkpoint & resume from it
# ============================================================
import os
import glob

def find_latest_checkpoint(models_path):
    """Trova il checkpoint con epoca più alta"""
    checkpoints = glob.glob(f"{models_path}/checkpoint_epoch_*.pt")
    if not checkpoints:
        return None, None

    # extract epoch number from the file
    epochs = []
    for cp in checkpoints:
        try:
            epoch = int(cp.split('epoch_')[-1].split('.pt')[0])
            epochs.append((epoch, cp))
        except:
            continue

    if not epochs:
        return None, None

    latest_epoch, latest_path = max(epochs, key=lambda x: x[0])
    return latest_path, latest_epoch

latest_cp, latest_epoch = find_latest_checkpoint(MODELS_PATH)

if latest_cp is not None:  # ← Controlla latest_cp, non solo "if latest_cp:"
    print(f"Checkpoint found for epoch {latest_epoch}: {latest_cp}")
    # ... carica checkpoint
else:
    print("No checkpoint found, restart from zero")
    trainer.epoch = 0

No checkpoint found, restart from zero


NameError: name 'trainer' is not defined

In [21]:
# ============================================================
# TRAINING MONITORING (excecute every time colab is re opened)
# ============================================================
import glob
import json
import os

def training_report(models_path):
    checkpoints = sorted(glob.glob(f"{models_path}/checkpoint_epoch_*.pt"))

    print("=" * 60)
    print("REPORT TRAINING")
    print("=" * 60)

    for cp in checkpoints:
        epoch = int(cp.split('epoch_')[-1].split('.pt')[0])
        size_mb = os.path.getsize(cp) / 1e6
        print(f"  Epoca {epoch}: {size_mb:.1f} MB")

    print(f"\nTotale checkpoint: {len(checkpoints)}")
    print(f"Ultimo completato: {len(checkpoints)-1}")

    # Calcola tempo stimato
    if len(checkpoints) >= 2:
        print(f"\nStima: ~{(len(checkpoints)) * 70} minuti totali investiti")

training_report(MODELS_PATH)

REPORT TRAINING

Totale checkpoint: 0
Ultimo completato: -1


In [22]:
'''
# Loads the last checkpoint saved
trainer.load_checkpoint(f"{MODELS_PATH}/checkpoint_epoch_0.pt")
'''
# Skip epochs already done
# trainer.epoch = 1
print(f"Resume from epoch: {trainer.epoch}")

print(f"\n{'='*50}")
print(f"START TRAINING")
print(f"Epochs: {N_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Checkpoint saved in: {MODELS_PATH}")
print(f"{'='*50}\n")

# For sanity check: 2-3 epoch on 10% data
# For completed training: 10-20 epochs

# To continue from a prev checkpoint:
# trainer.load_checkpoint(f"{MODELS_PATH}/checkpoint_epoch_3.pt")

trainer.fit(n_epochs=N_EPOCHS)

NameError: name 'trainer' is not defined

### Analysis of past training & parameters tuning
The first configuration of the project has the following parameters:

| `B_SIZE` | `MAX_SAMPLES` | `LR` | `warmup_steps` | `D_MODEL` | `D_FF` | `N_PARAMETERS` | `BATCH_PER_EPOCH` |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
|  16 | 1M | 1e-4 | 4000 | 512 | 2048 | 93M | 62.5K |

But the progression stucks arround `19%`, so:
- The `BATCH_SIZE` was changed from 16 to `8`.

This configuration, instead, stucked arround `49%`, requiring more parameters to be changed.

The second tuning of the parameters was the following:
| `B_SIZE` | `ACC_STEPS` | `MAX_SAMPLES` | `LR` | `warmup_steps` | `D_MODEL` | `D_FF` | `N_PARAMETERS` | `BATCH_PER_EPOCH` |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
|  4 | 4 | 200K | 1e-4 | 4000 | 512 | 2048 | 93M | 50K |

- Was added the `ACCUMULATION_STEPS`, in order to achieve a behaviour similar to the one achieved with `BATCH_SIZE` = `16`.
- The `MAX_SAMPLES` were reduced to `200K`.

With those changes, the `N_BATCH_PER_EPOCH` were reduced from 62.5K to `50K`, avoiding model overfitting. \
This configuration allows the model to finish the firt epoch, with a loss = `38.6`; passing to the execution of the `epoch 1` (freezed at `74%`) with its loss arround `35.4`.


The third configuration was the following:
| `B_SIZE` | `ACC_STEPS` | `MAX_SAMPLES` | `LR` | `warmup_steps` | `D_MODEL` | `D_FF` | `N_PARAMETERS` | `BATCH_PER_EPOCH` |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
|  4 | 4 | 100K | 5e-4 | 1000 | 256 | 1024 | 35.6M | 50K |

- Reduced again the `MAX_SAMPLES` to `100K`;
- The `warmup_steps` was changed to `1000`;
- The learning rate was `5e-4`;
- The `D_MODEL` was lowered to `256`;
- The `D_FF` lowered to `1024`.

But this configuration leaded to a structural underfitting, the model wasn't learning at all.

The final configuration was build on the past ones, avoiding mistakes done before.
| `B_SIZE` | `ACC_STEPS` | `MAX_SAMPLES` | `LR` | `warmup_steps` | `D_MODEL` | `D_FF` | `N_PARAMETERS` | `BATCH_PER_EPOCH` |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
|  4 | 4 | 100K | 2e-4 | 2000 | 512 | 2048 | 35.6M | 50K |

# TO ADD

### Final view
Here a final view of all the parameter tuning done across the multiple execution of the training.
| Parameters | START | Tuning 2 | Tuning 3 | FINAL |
| :--- | :--- | :--- | :--- | :--- |
| `BATCH_SIZE` | 16 → 8 | 4 | 4 | 4 |
| `ACCUMULATION_STEPS` | / | 4 | 4 | 4 |
| `MAX_SAMPLES` | 1M | 200k | 100k | 100k |
| `LR` | 1e-4 | 1e-4 | 5e-4 | 2e-4 |
| `warmup_steps` | 4000 | 4000 | 1000 | 2000 |
| `N_EPOCHS` | 10 | 10 | 10 | 5/6 |
| `D_MODEL` | 512 | 512 | 256 | 512 |
| `N_LAYERS` | 6 | 6 | 6 | 6 |
| `D_FF` | 2048 | 2048 | 1024 | 2048 |
| `N_PARAMETERS` | 93 M | 93 M | 35.6M | |
| `N_BATCH_PER_EPOCH` | 62.5K | 50K | 50K | |
| Training progress | stuck 19/49% | 1 epoch + 74% | 1 epoch + 78% | |
| Loss function | 9.8 | 38.6 → 35.4 | 39.7 → 37.4 | |

---
## 8. Translation test & Evaluation
*How can we know that the model is actually working?*

- **Inference:** A `translate()` function that takes raw English text and generates a Japanese translation using the trained weights.

In [ ]:
def translate(text, direction="en-jp"):
    model.eval()
    with torch.no_grad():
        sp_src = sp_en if direction == "en-jp" else sp_jp
        sp_tgt = sp_jp if direction == "en-jp" else sp_en

        src_ids = sp_src.encode(text, out_type=int, add_bos=True, add_eos=True)
        src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)

        out = model.translate(src_tensor, max_len=50, bos_id=2, eos_id=3)
        out_ids = [id for id in out[0].cpu().tolist() if id not in [0, 2, 3]]
        return sp_tgt.decode(out_ids)

# Test
test_sentences = [
    "Hello, how are you today?",
    "I love programming using Java.",
    "The weather was nice the other day.",
    "Thank you very much for your present.",
    "Where is the train station?"
]

print("=" * 50)
for sent in test_sentences:
    translated = translate(sent, "en-jp")
    print(f"EN: {sent}")
    print(f"JP: {translated}")
    print("-" * 50)

---
## 9. BLEU evaluation & pre-trained model

Automates the evaluation of `1,000 test phrases` using the **BLEU metric**, which provides a mathematical score for translation quality

In [ ]:
from src.evaluate import extract_test_set, evaluate_models

# Extract 1000 phrases for test model
test_en, test_jp, test_indices = extract_test_set(EN_FILE, JP_FILE, n=1000)

# Load best model
trainer.load_checkpoint(f"{MODELS_PATH}/best_model.pt")

# Evaluation EN -> JP
results_en_jp = evaluate_models(
    trainer.model,
    sp_en, sp_jp,
    test_en, test_jp,
    direction="en-jp"
    device=DEVICE
)

# Evaluation JP -> EN
results_jp_en = evaluate_models(
    trainer.model,
    sp_en, sp_jp,
    test_en, test_jp,
    direction="jp-en"
    device=DEVICE
)

# Save results
import json
with open(f"{MODELS_PATH}/evaluation_results.json", 'w') as f:
    json.dump({
        'en_jp': {
            'my_bleu': results_en_jp['my_bleu'],
            'pretrained_bleu': results_en_jp['pretrained_bleu']
        },
        'jp_en': {
            'my_bleu': results_jp_en['my_bleu'],
            'pretrained_bleu': results_jp_en['pretrained_bleu']
        }
    }, f, indent=2)

---
## 10. GUI on Colab

In [ ]:
# from src.gui import TranslatorApp

# app = TranslatorApp(trainer.model, sp_en, sp_jp, device='cuda')
# app.launch(share=True)  # Create temporary public link